# DCGAN para ArtBench-10

## Indice

- **1. Setup, dados e protocolo** - carregamento do ArtBench-10, transforms, seeds
- **2. Modelo DCGAN base** - arquitectura Generator/Discriminator
- **3. Melhorias manuais** - configuracao melhorada com TTUR, label smoothing, spectral norm
- **4. Grid Search** - optimizacao de hiperparametros (subset 20%)
- **5. Estudo de Ablacao** - analise de estabilidade (15 epocas)
- **6. Treino final** - dataset completo, 400 epocas
- **7. Avaliacao final** - 10 seeds, 5000 amostras, FID/KID rigoroso

### Resultados Finais
| Metrica | Valor |
|---------|-------|
| FID (5000 amostras, 10 seeds) | **25.39 +/- 1.53** |
| KID | **0.0118 +/- 0.0004** |
| Melhor Quick-FID (treino) | 141.02 |

## Setup do Ambiente Virtual e Dependencias

Para garantir reproducibilidade, o ideal e correr o notebook dentro de um ambiente virtual dedicado. As dependencias principais sao `torch`, `torchvision`, `torchmetrics[image]`, `datasets`, `matplotlib`, `pillow` e `tqdm`.

```bash
pip install torch torchvision matplotlib datasets pillow "torchmetrics[image]" tqdm
pip install ipywidgets
```

Este notebook assume ainda a existencia da pasta local `ArtBench-10` e do ficheiro `student_start_pack/training_20_percent.csv`, que define o subset oficial de desenvolvimento. Em VSCode, depois de criares o `.venv`, seleciona o kernel correto no canto superior direito antes de executar as celulas.


## 1. Setup, dados e protocolo

O protocolo experimental foi separado em duas fases para evitar leakage entre desenho do modelo e resultado final:

1. `dev_loader` e `dev_loader_aug` sao usados apenas na fase de desenvolvimento, grid search e ablacao, sempre com o subset oficial de 20%.
2. `full_train_loader` e `full_train_loader_aug` sao reservados ao treino final do modelo escolhido.
3. A avaliacao rigorosa usa multiplas repeticoes com seeds controladas e um numero fixo de amostras sinteticas.

A celula seguinte prepara todo o contexto tecnico: seeds, `device`, transforms com normalizacao para `[-1, 1]`, leitura local do ArtBench-10 e criacao dos `DataLoader`s base e aumentados. Esta organizacao e importante porque, em GANs, pequenas mudancas no preprocessing ou no split de referencia podem alterar significativamente o FID/KID.


In [ ]:
# Inline plotting is enabled by the notebook frontend.

# ==========================================
# SETUP CONSOLIDADO E COMPLETO (DCGAN)
# ==========================================
from __future__ import annotations
import sys
import json
import gc
import itertools
from tqdm.auto import tqdm
from torchvision.utils import save_image, make_grid
import random
import csv
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
import matplotlib.pyplot as plt

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance
except ImportError:
    print('AVISO: torchmetrics[image] não encontrado.')

# 1. Configurações de Reprodução e Caminhos
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'VAE', 'Diffusion_model', 'GAN', 'DAE', 'Exemplo'}:
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
KAGGLE_ROOT = PROJECT_ROOT / 'ArtBench-10'
TRAINING_CSV_PATH = PROJECT_ROOT / 'student_start_pack' / 'training_20_percent.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

import csv
import pickle
from datasets import Dataset as HFDataset, DatasetDict, Features, Image, ClassLabel

def _get_pickle_value(obj, key):
    if key in obj:
        return obj[key]
    bkey = key.encode("utf-8")
    if bkey in obj:
        return obj[bkey]
    raise KeyError(f"Missing key '{key}' in pickle object")



def _resolve_kaggle_paths(kaggle_root):
    root = Path(kaggle_root)
    csv_path = root / "ArtBench-10.csv"
    batch_dir = root / "artbench-10-python" / "artbench-10-batches-py"
    return root, csv_path, batch_dir



def load_kaggle_artbench10_splits(kaggle_root):
    root, csv_path, batch_dir = _resolve_kaggle_paths(kaggle_root)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Kaggle CSV not found: {csv_path}. "
            "Expected the original ArtBench-10 folder structure."
        )
    if not batch_dir.exists():
        raise FileNotFoundError(
            f"Kaggle CIFAR batches not found: {batch_dir}. "
            "Expected ArtBench-10/artbench-10-python/artbench-10-batches-py"
        )

    with open(batch_dir / "meta", "rb") as f:
        meta = pickle.load(f)
    styles = _get_pickle_value(meta, "styles")
    if not isinstance(styles, list) or len(styles) == 0:
        raise ValueError(f"Could not read class names from {batch_dir / 'meta'}")
    styles = [str(s).strip() for s in styles]
    style_to_id = {name: i for i, name in enumerate(styles)}

    csv_label_ids = {"train": {}, "test": {}}
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        required = {"split", "label", "cifar_index"}
        missing = required.difference(set(reader.fieldnames or []))
        if missing:
            raise ValueError(f"CSV is missing required columns {sorted(missing)}: {csv_path}")

        for row in reader:
            split = str(row.get("split", "")).strip().lower()
            if split not in csv_label_ids:
                continue

            label_name = str(row.get("label", "")).strip()
            if label_name not in style_to_id:
                raise ValueError(
                    f"Unknown label '{label_name}' in {csv_path}. "
                    f"Known labels: {styles}"
                )

            try:
                idx = int(row.get("cifar_index"))
            except Exception as exc:
                raise ValueError(f"Invalid cifar_index '{row.get('cifar_index')}' in {csv_path}") from exc

            csv_label_ids[split][idx] = int(style_to_id[label_name])

    def _load_batch(path):
        with open(path, "rb") as f:
            batch = pickle.load(f)
        data = np.asarray(_get_pickle_value(batch, "data"), dtype=np.uint8)
        labels = np.asarray(_get_pickle_value(batch, "labels"), dtype=np.int64)
        if data.ndim != 2 or data.shape[1] != 3072:
            raise ValueError(f"Unexpected data shape in {path}: {data.shape}")
        images = data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return images, labels

    train_images_chunks = []
    train_labels_chunks = []
    for batch_idx in range(1, 6):
        images, labels = _load_batch(batch_dir / f"data_batch_{batch_idx}")
        train_images_chunks.append(images)
        train_labels_chunks.append(labels)
    train_images = np.concatenate(train_images_chunks, axis=0)
    train_labels_raw = np.concatenate(train_labels_chunks, axis=0)
    test_images, test_labels_raw = _load_batch(batch_dir / "test_batch")

    def _labels_from_csv(split, n, labels_raw):
        ids = csv_label_ids[split]
        out = np.full((n,), -1, dtype=np.int64)
        for idx, label_id in ids.items():
            if idx < 0 or idx >= n:
                raise ValueError(
                    f"CSV {split} index {idx} out of bounds for {n} samples ({csv_path})"
                )
            out[idx] = int(label_id)
        missing = int(np.sum(out < 0))
        if missing > 0:
            raise ValueError(
                f"CSV {csv_path} is missing {missing} labels for split '{split}'."
            )
        mismatches = int(np.sum(out != labels_raw))
        if mismatches > 0:
            raise ValueError(
                f"CSV labels and batch labels disagree for {mismatches} samples in split '{split}'."
            )
        return out

    train_labels = _labels_from_csv("train", train_images.shape[0], train_labels_raw)
    test_labels = _labels_from_csv("test", test_images.shape[0], test_labels_raw)

    features = Features({
        "image": Image(),
        "label": ClassLabel(names=styles),
    })

    train_ds = HFDataset.from_dict(
        {
            "image": [train_images[i] for i in range(train_images.shape[0])],
            "label": train_labels.tolist(),
        },
        features=features,
    )
    test_ds = HFDataset.from_dict(
        {
            "image": [test_images[i] for i in range(test_images.shape[0])],
            "label": test_labels.tolist(),
        },
        features=features,
    )

    print(f"Dataset source: kaggle root='{root}'")
    return DatasetDict(train=train_ds, test=test_ds)



# 2. Definições de Dispositivo e Constantes
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 2

def safe_num_workers(requested: int) -> int:
    if 'ipykernel' in sys.modules and int(requested) > 0:
        return 0
    return int(requested)

# 3. Transforms (BASE e AUGMENTED)
BASE_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

AUGMENTED_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. Dataset e Funções de Utilidade
class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex['image']
        y = int(ex['label'])
        x = self.transform(img) if self.transform else img
        return x, y, real_idx

def load_ids_from_training_csv(csv_path: Path, index_column: str = 'train_id_original') -> list[int]:
    if not csv_path.exists():
        raise FileNotFoundError(f'Ficheiro não encontrado: {csv_path}')
    ids = []
    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        r = csv.DictReader(f)
        for row in r:
            v = str(row.get(index_column, '')).strip()
            if v: ids.append(int(v))
    return ids

def build_loader(indices, transform, shuffle=True, batch_size=BATCH_SIZE):
    ds = HFDatasetTorch(train_hf, transform=transform, indices=indices)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=safe_num_workers(NUM_WORKERS),
        pin_memory=torch.cuda.is_available(),
    )

def plot_loss_curves(history, title):
    plt.figure(figsize=(8, 4))
    for key, values in history.items():
        if values and isinstance(values[0], (int, float)):
            plt.plot(values, label=key)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 5. Carregar Dados Reais e Criar Loaders
print(f'Lendo ArtBench-10 de: {KAGGLE_ROOT}...')
try:
    hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
    train_hf = hf_ds['train']
    test_hf = hf_ds['test']
    class_names = list(train_hf.features['label'].names)

    subset_ids = load_ids_from_training_csv(TRAINING_CSV_PATH)
    full_train_ids = list(range(len(train_hf)))

    dev_loader = build_loader(subset_ids, BASE_TRANSFORM, shuffle=True)
    dev_loader_aug = build_loader(subset_ids, AUGMENTED_TRANSFORM, shuffle=True)
    full_train_loader = build_loader(full_train_ids, BASE_TRANSFORM, shuffle=True)
    full_train_loader_aug = build_loader(full_train_ids, AUGMENTED_TRANSFORM, shuffle=True)
    test_loader = DataLoader(
        HFDatasetTorch(test_hf, transform=BASE_TRANSFORM),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=safe_num_workers(NUM_WORKERS),
        pin_memory=torch.cuda.is_available(),
    )
    print('--- Setup Concluído com Sucesso ---')
except Exception as e:
    print(f'Erro ao carregar dados: {e}')
print('Dispositivo:', device)

# Funcao auxiliar para converter tensores de [-1, 1] para [0, 1]
def denormalize(t):
    return (t * 0.5 + 0.5).clamp(0, 1)



## 2. Modelo DCGAN base

A implementacao segue o espirito do DCGAN classico, mas ja incorpora algumas decisoes de engenharia para reduzir instabilidade:

**Generator**
- Parte de um vetor latente `z` e expande-o por convolucoes transpostas ate imagens RGB `32 x 32`.
- Usa `BatchNorm + ReLU` nos blocos interm?dios e `Tanh` na saida, ficando alinhado com a normalizacao dos dados para `[-1, 1]`.
- O tamanho do espaco latente e tratado como hiperparametro, porque influencia diretamente o compromisso entre diversidade e facilidade de treino.

**Discriminator**
- Recebe imagens RGB e comprime-as ate um logit escalar.
- Usa `Spectral Normalization` em todas as convolucoes para controlar melhor a Lipschitz-ness efetiva do discriminador.
- Mantem `LeakyReLU(0.2)` em todas as ativacoes para preservar gradientes em zonas negativas.

**Decisoes de estabilidade**
- `BCEWithLogitsLoss` como criterio adversarial.
- `AdamW` com `beta1 = 0.5` e `weight_decay = 1e-4`.
- `gradient clipping` com norma maxima `1.0`.
- `label smoothing` nas labels reais e possibilidade de atualizar o generator mais do que uma vez por iteracao (`n_critic`).

A ideia desta secao e estabelecer uma baseline suficientemente limpa para que as melhorias seguintes possam ser interpretadas como escolhas experimentais e nao como remendos arbitrarios.


In [ ]:
LATENT_DIM = 128


class Generator(nn.Module):
    """Generator com Tanh na saida (imagens em [-1,1])."""
    def __init__(self, latent_dim=LATENT_DIM, base_channels=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, base_channels * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1, bias=False),
            nn.Tanh(),  # saida em [-1,1], compativel com normalizacao dos dados
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """Discriminator com Spectral Normalization para estabilidade."""
    def __init__(self, base_channels=64):
        super().__init__()
        SN = nn.utils.spectral_norm
        self.net = nn.Sequential(
            SN(nn.Conv2d(3, base_channels, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            SN(nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            SN(nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            SN(nn.Conv2d(base_channels * 4, 1, 4, 1, 0, bias=False)),
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)


def init_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
        nn.init.normal_(module.weight.data, 0.0, 0.02)
        if getattr(module, 'bias', None) is not None:
            nn.init.zeros_(module.bias.data)


@torch.no_grad()
def sample_gan(generator, n_samples, latent_dim=None, seed=None):
    """Gera amostras e devolve em [0,1] (denormalizadas)."""
    if latent_dim is None:
        latent_dim = generator.latent_dim
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, latent_dim, 1, 1, generator=g, device=device)
    else:
        z = torch.randn(n_samples, latent_dim, 1, 1, device=device)
    generator.eval()
    out = generator(z)          # [-1,1]
    return denormalize(out)     # [0,1]


@torch.no_grad()
def quick_fid(generator, real_loader, n_samples=2000, seed=999):
    """FID rapido (2000 amostras) para usar como criterio de checkpoint."""
    if FrechetInceptionDistance is None:
        return float('inf')
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    seen = 0
    for real, _, _ in real_loader:
        real = real.to(device)
        batch = min(real.size(0), n_samples - seen)
        fid_metric.update(denormalize(real[:batch]), real=True)
        fid_metric.update(sample_gan(generator, batch, seed=seed), real=False)
        seen += batch
        if seen >= n_samples:
            break
    return float(fid_metric.compute().detach().cpu())


def train_dcgan(generator, discriminator, train_loader, *,
                epochs=20, lr_g=2e-4, lr_d=1e-4, beta1=0.5,
                label_smooth=0.9, run_name='dcgan_run',
                save_samples_every=5, fid_check_every=0,
                fid_loader=None, n_critic=1):
    """
    DCGAN com todas as melhorias:
    - Tanh + normalizacao [-1,1]
    - Spectral Norm no Discriminator
    - Label smoothing (real=label_smooth)
    - Two-timescale LR (lr_d < lr_g)
    - Gradient clipping
    - Generator treina n_critic vezes por update do Discriminator
    - AdamW com weight decay
    - Best FID checkpoint
    """
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    criterion = nn.BCEWithLogitsLoss()
    # AdamW tem weight decay, ligeiramente mais estavel que Adam
    opt_g = torch.optim.AdamW(generator.parameters(), lr=lr_g, betas=(beta1, 0.999), weight_decay=1e-4)
    opt_d = torch.optim.AdamW(discriminator.parameters(), lr=lr_d, betas=(beta1, 0.999), weight_decay=1e-4)
    history = {'g_loss': [], 'd_loss': [], 'fid_quick': []}
    best_fid = float('inf')

    for epoch in range(1, epochs + 1):
        generator.train()
        discriminator.train()
        total_g = total_d = 0.0
        progress = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False)

        for real, _, _ in progress:
            real = real.to(device)   # ja em [-1,1] pelo transform
            bsz = real.size(0)
            real_labels = label_smooth * torch.ones(bsz, 1, device=device)
            fake_labels = torch.zeros(bsz, 1, device=device)

            # --- Discriminator (1x por iteracao) ---
            opt_d.zero_grad(set_to_none=True)
            loss_real = criterion(discriminator(real), real_labels)
            noise = torch.randn(bsz, generator.latent_dim, 1, 1, device=device)
            fake = generator(noise)
            loss_fake = criterion(discriminator(fake.detach()), fake_labels)
            loss_d = 0.5 * (loss_real + loss_fake)
            loss_d.backward()
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
            opt_d.step()

            # --- Generator (n_critic vezes por iteracao) ---
            g_loss_sum = 0.0
            for _ in range(n_critic):
                opt_g.zero_grad(set_to_none=True)
                noise2 = torch.randn(bsz, generator.latent_dim, 1, 1, device=device)
                fake2 = generator(noise2)
                loss_g = criterion(discriminator(fake2), torch.ones(bsz, 1, device=device))
                loss_g.backward()
                torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
                opt_g.step()
                g_loss_sum += float(loss_g.item())
            avg_g_step = g_loss_sum / n_critic

            total_d += float(loss_d.item())
            total_g += avg_g_step
            progress.set_postfix(d=f'{loss_d.item():.4f}', g=f'{avg_g_step:.4f}')

        avg_d = total_d / max(1, len(train_loader))
        avg_g = total_g / max(1, len(train_loader))
        history['d_loss'].append(avg_d)
        history['g_loss'].append(avg_g)
        print(f'Epoch {epoch:03d}/{epochs} | d_loss={avg_d:.4f} | g_loss={avg_g:.4f}', end='')

        # Checkpoint por FID real
        if fid_check_every > 0 and fid_loader is not None and epoch % fid_check_every == 0:
            fid_val = quick_fid(generator, fid_loader)
            history['fid_quick'].append({'epoch': epoch, 'fid': fid_val})
            print(f' | FID_quick={fid_val:.2f}', end='')
            if fid_val < best_fid:
                best_fid = fid_val
                torch.save(generator.state_dict(), run_dir / 'generator_best.pt')
                torch.save(discriminator.state_dict(), run_dir / 'discriminator_best.pt')
                print(f' <- BEST', end='')
            generator.train()
        print()

        if epoch % save_samples_every == 0 or epoch == epochs:
            with torch.no_grad():
                samples = sample_gan(generator, 36, seed=2026)
                save_image(make_grid(samples.cpu(), nrow=6),
                           run_dir / f'samples_epoch_{epoch:03d}.png')

    torch.save(generator.state_dict(), run_dir / 'generator_last.pt')
    torch.save(discriminator.state_dict(), run_dir / 'discriminator_last.pt')
    best_ckpt = run_dir / 'generator_best.pt'
    if not best_ckpt.exists():
        torch.save(generator.state_dict(), best_ckpt)

    with open(run_dir / 'history.json', 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2)
    return history, run_dir


# --- Treino base (subset 20%, sem FID check) ---
BASE_CONFIG = {'base_channels': 64, 'epochs': 20, 'lr_g': 2e-4, 'lr_d': 1e-4, 'beta1': 0.5}
base_G = Generator(base_channels=BASE_CONFIG['base_channels']).to(device)
base_D = Discriminator(base_channels=BASE_CONFIG['base_channels']).to(device)
base_G.apply(init_weights)
base_D.apply(init_weights)
base_history, base_dir = train_dcgan(
    base_G, base_D, dev_loader,
    epochs=BASE_CONFIG['epochs'],
    lr_g=BASE_CONFIG['lr_g'],
    lr_d=BASE_CONFIG['lr_d'],
    beta1=BASE_CONFIG['beta1'],
    run_name='dcgan_base_dev_artbench10',
    save_samples_every=5,
    n_critic=2,  # G treina 2x por update do D
)
plot_loss_curves(base_history, 'DCGAN base - dev subset')
grid_base = make_grid(sample_gan(base_G, 36, seed=2026).cpu(), nrow=6)
plt.figure(figsize=(8, 8))
plt.imshow(np.clip(grid_base.permute(1, 2, 0).numpy(), 0, 1))
plt.axis('off')
plt.title('Amostras DCGAN - base (Tanh, G x2)')
plt.show()

## 3. Melhorias manuais no DCGAN

Antes da grid search formal, fazemos uma etapa curta de refinamento manual para confirmar que a arquitetura base consegue beneficiar de mais capacidade e de um esquema de treino assimetrico.

As principais alteracoes testadas aqui sao:
- aumento do numero de canais base para `96`, de forma a dar mais largura ao modelo em texturas artisticas complexas;
- `TTUR` impl?cito via `lr_G > lr_D`, reduzindo a probabilidade de o discriminador dominar demasiado cedo;
- validacao qualitativa por grelhas de amostras e validacao quantitativa por `quick FID`.

Esta etapa nao substitui a busca sistematica, mas serve para verificar rapidamente se estamos numa regiao promissora do espaco de hiperparametros antes de gastar tempo de GPU no subset de desenvolvimento.


In [ ]:
IMPROVED_CONFIG = {
    'base_channels': 96,
    'epochs': 30,
    'lr_g': 2e-4,
    'lr_d': 8e-5,
    'beta1': 0.5,
}
improved_G = Generator(base_channels=IMPROVED_CONFIG['base_channels']).to(device)
improved_D = Discriminator(base_channels=IMPROVED_CONFIG['base_channels']).to(device)
improved_G.apply(init_weights)
improved_D.apply(init_weights)
improved_history, improved_dir = train_dcgan(
    improved_G, improved_D, dev_loader_aug,
    epochs=IMPROVED_CONFIG['epochs'],
    lr_g=IMPROVED_CONFIG['lr_g'],
    lr_d=IMPROVED_CONFIG['lr_d'],
    beta1=IMPROVED_CONFIG['beta1'],
    run_name='dcgan_improved_dev_artbench10',
    save_samples_every=5,
    fid_check_every=10,
    fid_loader=test_loader,
    n_critic=2,
)
plot_loss_curves(improved_history, 'DCGAN melhorado - dev subset')
grid_imp = make_grid(sample_gan(improved_G, 36, seed=2026).cpu(), nrow=6)
plt.figure(figsize=(8, 8))
plt.imshow(np.clip(grid_imp.permute(1, 2, 0).numpy(), 0, 1))
plt.axis('off')
plt.title('Amostras DCGAN - melhorado (96ch, Tanh, G x2)')
plt.show()

## 4. Otimizacao de Hiperparametros: Grid Search (Subset 20%)

A grid search e executada apenas no subset oficial de 20%, com poucas epocas por configuracao, para reduzir custo computacional mantendo um criterio coerente de selecao.

**Espaco de procura**
- `base_channels`: `[64, 96]`
- `lr_g`: `[2e-4, 1.5e-4]`
- `latent_dim`: `[100, 128]`

Isto gera 8 configuracoes candidatas. A selecao nao depende apenas da qualidade visual de uma grelha final: observamos tambem a aproximacao ao equilibrio adversarial e a estabilidade das perdas.

Importa notar que o `quick FID` usado aqui funciona como **metrica direcional de checkpointing**. Como e calculado com um protocolo mais leve do que a avaliacao final, ele e util para ordenar checkpoints e perceber tendencias, mas nao deve ser confundido com a metrica oficial do relatorio.


In [ ]:
import itertools

RUN_GRID_SEARCH = True
GRID_EPOCHS = 15
GRID_SEARCH_SPACE = {
    'base_channels': [64, 96],
    'lr_g': [2e-4, 1.5e-4],
    'latent_dim': [100, 128],
}


def run_grid_search(space, epochs=GRID_EPOCHS):
    keys = list(space.keys())
    results = []
    for values in itertools.product(*[space[k] for k in keys]):
        params = dict(zip(keys, values))
        G = Generator(latent_dim=params['latent_dim'], base_channels=params['base_channels']).to(device)
        D = Discriminator(base_channels=params['base_channels']).to(device)
        G.apply(init_weights)
        D.apply(init_weights)
        run_tag = '_'.join(f'{k}-{v}' for k, v in params.items())
        history, run_dir = train_dcgan(
            G, D, dev_loader_aug,
            epochs=epochs,
            lr_g=params['lr_g'],
            lr_d=params['lr_g'] * 0.5,
            beta1=0.5,
            run_name='grid_' + run_tag,
            save_samples_every=epochs,
            n_critic=2,
        )
        score = abs(history['d_loss'][-1] - 0.5) + history['g_loss'][-1]
        results.append({'params': params, 'score': score, 'run_dir': str(run_dir)})
        print('Resultado:', results[-1])
    return sorted(results, key=lambda item: item['score'])


if RUN_GRID_SEARCH:
    grid_results = run_grid_search(GRID_SEARCH_SPACE)
    best_grid_result = grid_results[0]
    print('Melhor config:', best_grid_result)
    
    # Tabela comparativa dos resultados da Grid Search
    print('\n=== TABELA GRID SEARCH ===')
    print(f'{"Config":<45} | {"Score":<10}')
    print('-' * 60)
    for r in grid_results:
        p = r['params']
        tag = f"ch={p['base_channels']}, lr={p['lr_g']}, z={p['latent_dim']}"
        print(f'{tag:<45} | {r["score"]:.6f}')
else:
    grid_results = []
    best_grid_result = None
    print('Grid search desativada. Define RUN_GRID_SEARCH = True para executar.')

# --- Visualizar resultados da Grid Search ---
if grid_results:
    print('\n=== RESULTADOS GRID SEARCH ===')
    for r in sorted(grid_results, key=lambda x: x['score']):
        print(f"  base={r['params']['base_channels']}, lr_g={r['params']['lr_g']}, "
              f"latent={r['params']['latent_dim']} -> Score={r['score']:.4f}")

    fig, ax = plt.subplots(figsize=(8, 5))
    names = [f"b{r['params']['base_channels']}_lr{r['params']['lr_g']}_z{r['params']['latent_dim']}" for r in grid_results]
    scores = [r['score'] for r in grid_results]

    ax.plot(range(len(scores)), scores, 'o-', color='#e67e22', linewidth=2, markersize=6)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=8, rotation=45, ha='right')
    ax.set_ylabel('Stability Score (lower is better)')
    ax.set_title('Grid Search: Estabilidade')
    ax.grid(True, alpha=0.3)

    plt.suptitle('DCGAN Grid Search Results (20% subset, 15 epochs)', fontsize=13)
    plt.tight_layout()
    plt.show()


## 5. Estudo de Ablacao de Estabilidade

A ablacao tem um objetivo metodologico claro: mostrar que o comportamento estavel do DCGAN nao surge por sorte, mas pela combinacao das tecnicas introduzidas ao longo do notebook.

As quatro variantes testadas sao:
1. **V1 - Vanilla**: sem TTUR, sem label smoothing e sem reforco no generator.
2. **V2 - +TTUR**: apenas desequilibrio controlado das learning rates.
3. **V3 - +TTUR + Label Smoothing**: reduzimos a overconfidence do discriminador.
4. **V4 - Full**: adicionamos tambem `n_critic = 2`, permitindo ao generator responder mais agressivamente.

O que nos interessa aqui nao e apenas qual variante tem a menor loss final, mas sim qual delas aproxima as dinamicas do equilibrio de Nash de forma consistente. Em problemas artistico-visuais como o ArtBench-10, essa consistencia vale tanto quanto uma melhoria marginal de FID, porque reduz o risco de colapso ou de divergencia tardia.


In [ ]:
print("--- ESTUDO DE ABLAÇÃO DE ESTABILIDADE (15 ÉPOCAS) ---")
ABLATION_EPOCHS = 15

# 1. Vanilla (Sem defesas)
print("\n[Ablação 1] GAN Vanilla (Sem Defesas)")
ab1_G = Generator(base_channels=64).to(device)
ab1_D = Discriminator(base_channels=64).to(device)
ab1_G.apply(init_weights)
ab1_D.apply(init_weights)
ab1_hist, ab1_dir = train_dcgan(
    ab1_G, ab1_D, dev_loader, epochs=ABLATION_EPOCHS,
    lr_g=2e-4, lr_d=2e-4,  # Identicos
    label_smooth=1.0,      # Desligado
    n_critic=1,            # Desligado
    run_name='ablation_1_vanilla'
)

# 2. TTUR apenas
print("\n[Ablação 2] Apenas TTUR (lr_d = lr_g * 0.5)")
ab2_G = Generator(base_channels=64).to(device)
ab2_D = Discriminator(base_channels=64).to(device)
ab2_G.apply(init_weights)
ab2_D.apply(init_weights)
ab2_hist, ab2_dir = train_dcgan(
    ab2_G, ab2_D, dev_loader, epochs=ABLATION_EPOCHS,
    lr_g=2e-4, lr_d=1e-4,  # TTUR Ligado
    label_smooth=1.0,
    n_critic=1,
    run_name='ablation_2_ttur'
)

# 3. TTUR + Label Smoothing
print("\n[Ablação 3] TTUR + Label Smoothing (0.9)")
ab3_G = Generator(base_channels=64).to(device)
ab3_D = Discriminator(base_channels=64).to(device)
ab3_G.apply(init_weights)
ab3_D.apply(init_weights)
ab3_hist, ab3_dir = train_dcgan(
    ab3_G, ab3_D, dev_loader, epochs=ABLATION_EPOCHS,
    lr_g=2e-4, lr_d=1e-4,
    label_smooth=0.9,      # Ligado
    n_critic=1,
    run_name='ablation_3_ttur_ls'
)

# 4. Full (V4)
print("\n[Ablação 4] Todas as Defesas (TTUR, LS, n_critic=2)")
ab4_G = Generator(base_channels=64).to(device)
ab4_D = Discriminator(base_channels=64).to(device)
ab4_G.apply(init_weights)
ab4_D.apply(init_weights)
ab4_hist, ab4_dir = train_dcgan(
    ab4_G, ab4_D, dev_loader, epochs=ABLATION_EPOCHS,
    lr_g=2e-4, lr_d=1e-4,
    label_smooth=0.9,
    n_critic=2,            # Ligado
    run_name='ablation_4_full'
)

# Plotting the discriminator loss across the 4 runs
plt.figure(figsize=(10,6))
plt.plot(ab1_hist['d_loss'], label='V1: Vanilla')
plt.plot(ab2_hist['d_loss'], label='V2: + TTUR')
plt.plot(ab3_hist['d_loss'], label='V3: + TTUR + LS')
plt.plot(ab4_hist['d_loss'], label='V4: Full (TTUR+LS+n_critic=2)', linewidth=2.5)
plt.axhline(0.693, color='r', linestyle='--', label='Equilíbrio Matemático (0.693)')
plt.title('Estudo de Ablação: Estabilidade da D_loss')
plt.xlabel('Época')
plt.ylabel('Loss do Discriminador')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Estudo de Ablação Concluído!")

## 6. Treino final DCGAN no conjunto completo (400 epocas)

Configuracao final: `base_channels=64`, `latent_dim=128`, `lr_g=1.5e-4`, `lr_d=1e-4`, `beta1=0.5`, `n_critic=2`.

O dataset e pre-carregado em RAM para velocidade maxima (~10 seg/epoch). O treino longo confirmou que, embora as losses estabilizem cedo perto do equilibrio adversarial, o `quick FID` continuou a melhorar ate a epoca 400.

In [ ]:
# ============================================================
# TREINO FINAL DCGAN - Dataset completo, 400 epocas
# ============================================================
from tqdm.auto import tqdm
import gc
import torchvision.transforms.functional as TF

FINAL_CONFIG = {
    'base_channels': 64,
    'latent_dim': 128,
    'lr_g': 1.5e-4,
    'lr_d': 1e-4,
    'beta1': 0.5,
    'epochs': 400,
}

# --- Cache dados BASE em RAM, augmentacao on-the-fly ---
print('Caching dataset em RAM (dados base)...')
_base_x, _base_y = [], []
for _x, _y, _ in tqdm(full_train_loader, desc='Cache'):
    _base_x.append(_x)
    _base_y.append(_y)
_base_x = torch.cat(_base_x)
_base_y = torch.cat(_base_y)
print(f'  {len(_base_x)} imagens cached.')

class AugmentedTensorDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        img = self.x[i]
        if torch.rand(1).item() < 0.5:
            img = TF.hflip(img)
        return img, self.y[i], i

class TensorTripleDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i], i

fast_train_loader = torch.utils.data.DataLoader(
    AugmentedTensorDataset(_base_x, _base_y),
    batch_size=128, shuffle=True, drop_last=True, num_workers=0,
)
fast_eval_loader = torch.utils.data.DataLoader(
    TensorTripleDataset(_base_x, _base_y),
    batch_size=128, shuffle=False, num_workers=0,
)
print('Loaders prontos (augmentacao on-the-fly).')

final_G = Generator(latent_dim=FINAL_CONFIG['latent_dim'], base_channels=FINAL_CONFIG['base_channels']).to(device)
final_D = Discriminator(base_channels=FINAL_CONFIG['base_channels']).to(device)
final_G.apply(init_weights)
final_D.apply(init_weights)

final_history, final_dir = train_dcgan(
    final_G, final_D, fast_train_loader,
    epochs=FINAL_CONFIG['epochs'],
    lr_g=FINAL_CONFIG['lr_g'],
    lr_d=FINAL_CONFIG['lr_d'],
    beta1=FINAL_CONFIG['beta1'],
    run_name='dcgan_final_full_artbench10',
    save_samples_every=10,
    fid_check_every=10,
    fid_loader=fast_eval_loader,
    n_critic=2,
)

plot_loss_curves(final_history, 'DCGAN Final Training (400 epocas)')
print(f'\nTreino concluido! Modelo guardado em: {final_dir}')


# --- GRAFICOS DE TREINO (inline) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(final_history['d_loss'], label='D Loss', color='#e74c3c', alpha=0.8)
axes[0].plot(final_history['g_loss'], label='G Loss', color='#3498db', alpha=0.8)
axes[0].axhline(0.693, color='black', linestyle='--', alpha=0.5, label='Nash (0.693)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('DCGAN Training Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

fid_data = final_history.get('fid_quick', [])
if fid_data:
    if isinstance(fid_data[0], dict):
        fep = [x['epoch'] for x in fid_data]
        fval = [x['fid'] for x in fid_data]
    else:
        fep = list(range(10, 10*len(fid_data)+1, 10))
        fval = fid_data
    best_idx = fval.index(min(fval))
    axes[1].plot(fep, fval, 'o-', color='#2ecc71', markersize=3)
    axes[1].scatter([fep[best_idx]], [fval[best_idx]], color='red', s=100, zorder=5,
                    label=f'Best: {min(fval):.2f} (ep {fep[best_idx]})')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FID')
    axes[1].set_title('Quick-FID Evolution')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].hist(final_history['d_loss'][-50:], bins=20, color='#e74c3c', alpha=0.7, label='D Loss')
axes[2].hist(final_history['g_loss'][-50:], bins=20, color='#3498db', alpha=0.7, label='G Loss')
axes[2].axvline(0.693, color='black', linestyle='--', label='Nash')
axes[2].set_title('Loss Distribution (last 50 epochs)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('DCGAN - Analise de Treino', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'dcgan_final_full_artbench10' / 'dcgan_training_analysis.png'), dpi=150)
plt.show()

# --- AMOSTRAS GERADAS (inline) ---
# Load best checkpoint
best_G = Generator(latent_dim=FINAL_CONFIG['latent_dim'], base_channels=FINAL_CONFIG['base_channels']).to(device)
best_G.load_state_dict(torch.load(final_dir / 'generator_best.pt', map_location=device, weights_only=True))
best_G.eval()

fig, axes = plt.subplots(10, 8, figsize=(16, 20))
with torch.no_grad():
    z = torch.randn(80, FINAL_CONFIG['latent_dim'], 1, 1, device=device)
    imgs = denormalize(best_G(z)).cpu()
    for i in range(80):
        r, c = divmod(i, 8)
        axes[r, c].imshow(imgs[i].permute(1, 2, 0).numpy())
        axes[r, c].axis('off')

plt.suptitle('DCGAN - Amostras Geradas (80 imagens, best checkpoint)', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'dcgan_final_full_artbench10' / 'dcgan_samples_grid.png'), dpi=150)
plt.show()

best_fid = min([x['fid'] for x in final_history.get('fid_quick', [{'fid': float('inf')}])] or [float('inf')])
print(f'\nMelhor FID durante treino: {best_fid:.2f}')


## 7. Avaliacao Final DCGAN

A celula seguinte carrega o melhor checkpoint guardado durante o treino e corre a avaliacao rigorosa com **10 seeds** e **5000 amostras** cada. Pode ser re-executada independentemente sem repetir o treino.

**Resultado: FID = 25.39 +/- 1.53 | KID = 0.0118 +/- 0.0004**

In [ ]:
# ============================================================
# AVALIACAO FINAL DCGAN (10 seeds, 5000 amostras)
# Pode ser re-executada independentemente do treino
# ============================================================
import numpy as np

FINAL_CONFIG = {
    'base_channels': 64,
    'latent_dim': 128,
    'lr_g': 1.5e-4,
    'lr_d': 1e-4,
    'beta1': 0.5,
    'epochs': 400,
}

final_dir = OUTPUT_ROOT / 'dcgan_final_full_artbench10'

# --- 1. GRAFICOS DE TREINO ---
history = json.loads((final_dir / 'history.json').read_text('utf-8'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['d_loss'], label='D Loss', color='#e74c3c', alpha=0.8)
axes[0].plot(history['g_loss'], label='G Loss', color='#3498db', alpha=0.8)
axes[0].axhline(0.693, color='black', linestyle='--', alpha=0.5, label='Nash (0.693)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('DCGAN Training Loss (400 epochs)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

fid_data = history.get('fid_quick', [])
if fid_data and isinstance(fid_data[0], dict):
    fid_epochs = [x['epoch'] for x in fid_data]
    fid_vals = [x['fid'] for x in fid_data]
else:
    fid_epochs = list(range(10, 10*len(fid_data)+1, 10))
    fid_vals = fid_data
best_idx = fid_vals.index(min(fid_vals))
axes[1].plot(fid_epochs, fid_vals, 'o-', color='#2ecc71', markersize=4)
axes[1].scatter([fid_epochs[best_idx]], [fid_vals[best_idx]], color='red', s=100, zorder=5,
                label=f'Best: {min(fid_vals):.2f} (ep {fid_epochs[best_idx]})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FID (2000 samples)')
axes[1].set_title('Quick-FID Evolution')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].hist(history['d_loss'][-50:], bins=20, color='#e74c3c', alpha=0.7, label='D Loss (last 50)')
axes[2].hist(history['g_loss'][-50:], bins=20, color='#3498db', alpha=0.7, label='G Loss (last 50)')
axes[2].axvline(0.693, color='black', linestyle='--', label='Nash')
axes[2].set_title('Loss Distribution (last 50 epochs)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('DCGAN - Analise de Treino', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(final_dir / 'dcgan_training_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- 2. AMOSTRAS GERADAS (melhor checkpoint) ---
print('\n=== AMOSTRAS GERADAS (Melhor Checkpoint) ===')
best_G = Generator(latent_dim=FINAL_CONFIG['latent_dim'],
                    base_channels=FINAL_CONFIG['base_channels']).to(device)
best_G.load_state_dict(torch.load(
    final_dir / 'generator_best.pt', map_location=device, weights_only=True))
best_G.eval()

fig, axes = plt.subplots(10, 8, figsize=(16, 20))
with torch.no_grad():
    z = torch.randn(80, FINAL_CONFIG['latent_dim'], 1, 1, device=device)
    imgs = denormalize(best_G(z)).cpu()
    for i in range(80):
        r, c = divmod(i, 8)
        axes[r, c].imshow(imgs[i].permute(1, 2, 0).numpy())
        axes[r, c].axis('off')

plt.suptitle('DCGAN - Amostras Geradas (80 imagens, best checkpoint)', fontsize=14)
plt.tight_layout()
plt.savefig(str(final_dir / 'dcgan_samples_grid.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- 3. AVALIACAO RIGOROSA (10 seeds x 5000) ---
eval_file = final_dir / 'evaluation_5000_samples_10_repeats.json'
if eval_file.exists():
    eval_results = json.loads(eval_file.read_text('utf-8'))
    print('\nResultados carregados de avaliacao existente.')
else:
    print('\n=== AVALIACAO RIGOROSA (10 seeds x 5000 amostras) ===')
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance

    eval_results = {'fid': [], 'kid': []}
    for seed in range(10):
        print(f'Seed {seed}/9...', end=' ')
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

        fid_m = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        kid_m = KernelInceptionDistance(feature=2048, normalize=True, subset_size=100, subsets=50).to(device)

        generated = 0
        while generated < 5000:
            bsz = min(256, 5000 - generated)
            z = torch.randn(bsz, FINAL_CONFIG['latent_dim'], 1, 1, device=device)
            with torch.no_grad():
                fake = denormalize(best_G(z))
            fid_m.update(fake, real=False)
            kid_m.update(fake, real=False)
            generated += bsz

        seen = 0
        for real, _, _ in full_train_loader:
            real = denormalize(real.to(device))
            bsz = min(real.size(0), 5000 - seen)
            fid_m.update(real[:bsz], real=True)
            kid_m.update(real[:bsz], real=True)
            seen += bsz
            if seen >= 5000: break

        fid_val = float(fid_m.compute().detach().cpu())
        kid_val = float(kid_m.compute()[0].detach().cpu())
        print(f'FID={fid_val:.2f}, KID={kid_val:.4f}')
        eval_results['fid'].append(fid_val)
        eval_results['kid'].append(kid_val)

    with open(eval_file, 'w') as f:
        json.dump(eval_results, f, indent=2)

# --- 4. GRAFICOS DE AVALIACAO ---
fid_arr = np.array(eval_results['fid'])
kid_arr = np.array(eval_results['kid'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(range(len(fid_arr)), fid_arr, 'o-', color='#2ecc71', linewidth=2, markersize=6)
axes[0].axhline(fid_arr.mean(), color='red', linestyle='--',
                label=f'Mean: {fid_arr.mean():.2f} +/- {fid_arr.std():.2f}')
axes[0].set_xlabel('Seed'); axes[0].set_ylabel('FID')
axes[0].set_title('FID per Seed (5000 samples)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(range(len(kid_arr)), kid_arr, 'o-', color='#e67e22', linewidth=2, markersize=6)
axes[1].axhline(kid_arr.mean(), color='red', linestyle='--',
                label=f'Mean: {kid_arr.mean():.4f} +/- {kid_arr.std():.4f}')
axes[1].set_xlabel('Seed'); axes[1].set_ylabel('KID')
axes[1].set_title('KID per Seed (5000 samples)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('DCGAN - Avaliacao Rigorosa', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(final_dir / 'dcgan_eval_per_seed.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- 5. RESULTADO FINAL ---
print(f'\n{"="*50}')
print(f'  DCGAN RESULTADO FINAL')
print(f'  FID: {fid_arr.mean():.2f} +/- {fid_arr.std():.2f}')
print(f'  KID: {kid_arr.mean():.4f} +/- {kid_arr.std():.4f}')
print(f'{"="*50}')


## Notas Finais

Este notebook mostra que o DCGAN consegue aprender a distribuicao visual do ArtBench-10 de forma muito eficaz:

- **FID = 25.39 +/- 1.53** - melhor resultado global entre os modelos deste conjunto de experiencias
- **KID = 0.0118 +/- 0.0004** - distribuicao muito proxima dos dados reais
- **Melhor Quick-FID = 141.02** na epoca 400, mostrando melhoria mesmo depois da estabilizacao das losses
- A arquitectura com spectral normalization, TTUR, label smoothing e `n_critic=2` garante estabilidade de treino
- O modelo produz imagens artisticas diversas e reconheciveis, embora sem controlo explicito por classe